In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [3]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [5]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データをもとにいくつかの傾向を分析します。以下のポイントを考慮しながら、販売パフォーマンスを評価します。

### 1. カテゴリー別の売上分析
まず、カテゴリー別に売上を集計することで、どのカテゴリーが最も売上を上げているかを確認します。

- **食品**
- **衣服**
- **日用品**

それぞれのカテゴリーについて、売上や数量、原価の平均、合計を計算します。例えば、食品が最も高い売上を上げている場合、プロモーションや新商品導入の計画を考えるべきです。

### 2. 商品コード・商品名別の効果分析
販売データを商品別に集計し、どの商品の売れ行きが良いか特定します。特に、以下の指標が重要です。

- **売上合計**
- **数量合計**
- **利益（売上 - 原価）**

これにより、人気商品や利益率が高い商品を特定し、在庫管理やマーケティング戦略を立てやすくなります。

### 3. 売上の時間的トレンド
売上日ごとのトレンドを確認し、月別あるいは週別で売上がどのように変動しているかを視覚化することで、季節要因やプロモーションの効果を把握します。例えば、特定の月に売上が上昇している場合、過去のプロモーションやイベントとの関連性を調査します。

### 4. 利益率の分析
各商品の利益を計算し、どの商品の利益率が高いかを評価します。利益率が高い商品は重点的にプロモーションを行うべきです。

### 5. その他の要因の考慮
- **価格変動**：単価の変動が売上に与える影響を分析し、価格戦略の改善点を探ります。
- **競合分析**：同じカテゴリにおける競合製品の動向を追跡し、価格やプロモーションの改善に役立てます。

### まとめ
上記の分析を通じて、企業はデータ駆動の意思決定を行い、効果的なマーケティング戦略を計画し、実行することができるようになります。さらに、優先すべき商品の特定や、需要の高い時期に合わせたプロモーション活動を行うことで、売上向上が見込めます。具体的なデータの数値を取り入れたより詳細な分析を行うことが可能ですので、必要に応じて追加のデータや詳細をお勧めします。


In [6]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0   売上データをもとにいくつかの傾向を分析します。以下のポイントを考慮しながら、販売パフォーマン...
1                                                    
2                                  ### 1. カテゴリー別の売上分析
3     まず、カテゴリー別に売上を集計することで、どのカテゴリーが最も売上を上げているかを確認します。
4                                                    
5                                            - **食品**
6                                            - **衣服**
7                                           - **日用品**
8                                                    
9   それぞれのカテゴリーについて、売上や数量、原価の平均、合計を計算します。例えば、食品が最も高...
10                                                   
11                             ### 2. 商品コード・商品名別の効果分析
12    販売データを商品別に集計し、どの商品の売れ行きが良いか特定します。特に、以下の指標が重要です。
13                                                   
14                                         - **売上合計**
15                                         - **数量合計**
16                                  - **利益（売上 - 原価）**
17                          

In [7]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

In [8]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。
